# Preparación del dataset — Vocales LESCO

Convierte y redimensiona las fotos a 128×128 a color y genera el ZIP del dataset.

## 1. Instalar dependencias e importar

In [ ]:
!pip install -q pillow-heif

import os, io, shutil, zipfile
from PIL import Image, ImageOps
from pillow_heif import register_heif_opener
from google.colab import files
import matplotlib.pyplot as plt

register_heif_opener()   # permite abrir fotos .heic del iPhone
print("Listo. Soporte HEIC activado.")


## 2. Configuración

In [ ]:
CLASES   = ["A", "E", "I", "O", "U"]   # <-- cambiá si usás otras señas
IMG_SIZE = (128, 128)
BASE     = "/content/dataset"          # aquí se guardan las fotos ya procesadas

# CROP_CUADRADO: True recorta al centro (sin deformar); False redimensiona directo
CROP_CUADRADO = False

PREVIEW_POR_CLASE = 3   # cuántos ejemplos (original vs 128x128) guardar por clase para control
EXT = (".jpg", ".jpeg", ".png", ".heic", ".heif", ".bmp", ".webp")

os.makedirs(BASE, exist_ok=True)
_previews = []   # se llena solo durante el procesamiento
print("Clases:", CLASES, "| tamaño:", IMG_SIZE)


## 3. Función de procesamiento

In [ ]:
def procesar(clase, archivos):
    destino = os.path.join(BASE, clase)
    os.makedirs(destino, exist_ok=True)
    ya = len(os.listdir(destino))          # continuar numeración si ya había fotos
    n = 0
    for nombre, contenido in archivos.items():
        try:
            original = Image.open(io.BytesIO(contenido))
            original = ImageOps.exif_transpose(original)   # corrige orientación del celular
            img = original.convert("RGB")                  # asegura color (3 canales)
            if CROP_CUADRADO:
                img = ImageOps.fit(img, IMG_SIZE)          # recorta al centro, sin deformar
            else:
                img = img.resize(IMG_SIZE)                 # redimensiona directo a 128x128
            n += 1
            img.save(os.path.join(destino, f"{clase}_{ya+n:03d}.png"))

            if sum(1 for p in _previews if p[0] == clase) < PREVIEW_POR_CLASE:
                orig_thumb = original.convert("RGB").copy()
                orig_thumb.thumbnail((256, 256))           # se reduce solo para mostrar
                _previews.append((clase, orig_thumb, img.copy()))
        except Exception as e:
            print(f"   ⚠ no pude procesar '{nombre}': {e}")
    print(f"   ✔ {clase}: {n} fotos nuevas (total {ya+n})")


## 4A. Opción A — Subir las fotos clase por clase

In [ ]:
for clase in CLASES:
    print(f"\n===== Subí las fotos de la seña '{clase}' =====")
    archivos = files.upload()
    procesar(clase, archivos)

print("\n¡Todas las clases procesadas!")


## 4B. Opción B — Subir un ZIP ya organizado

In [ ]:
subida = files.upload()                       # elegí tu ZIP con las carpetas por clase
nombre_zip = list(subida.keys())[0]

crudo = "/content/_crudo"
shutil.rmtree(crudo, ignore_errors=True)
with zipfile.ZipFile(nombre_zip, "r") as z:
    z.extractall(crudo)

# Ignorar la carpeta basura de Mac si aparece
shutil.rmtree(os.path.join(crudo, "__MACOSX"), ignore_errors=True)

# Detectar la carpeta que contiene DIRECTAMENTE las subcarpetas de clase
subdirs = [d for d in os.listdir(crudo)
           if os.path.isdir(os.path.join(crudo, d)) and not d.startswith(".")]
if len(subdirs) == 1 and subdirs[0] not in CLASES:
    raiz = os.path.join(crudo, subdirs[0])    # había un nivel extra (ej. dataset/)
else:
    raiz = crudo
print("Carpeta raíz detectada:", raiz)
print("Subcarpetas encontradas:", sorted(subdirs if raiz == crudo else os.listdir(raiz)))

# Mapa tolerante a mayúsculas/minúsculas: 'a' -> 'A', etc.
disponibles = {d.lower(): d for d in os.listdir(raiz)
               if os.path.isdir(os.path.join(raiz, d))}

for clase in CLASES:
    if clase.lower() not in disponibles:
        print(f"   ⚠ no encontré la carpeta de '{clase}', la salto")
        continue
    carpeta = os.path.join(raiz, disponibles[clase.lower()])
    archivos = {}
    for nombre in sorted(os.listdir(carpeta)):
        if nombre.lower().endswith(EXT):
            with open(os.path.join(carpeta, nombre), "rb") as f:
                archivos[nombre] = f.read()
    procesar(clase, archivos)

print("\n¡ZIP procesado!")


## 5. Control de conversión (original vs 128×128)

In [ ]:
if not _previews:
    print("Todavía no hay ejemplos. Corré la sección 4A o 4B primero.")
else:
    filas = len(_previews)
    plt.figure(figsize=(6, 2.6 * filas))
    for i, (clase, orig, trans) in enumerate(_previews):
        plt.subplot(filas, 2, 2*i + 1)
        plt.imshow(orig); plt.axis("off")
        plt.title(f"{clase} — original ({orig.width}×{orig.height})", fontsize=9)
        plt.subplot(filas, 2, 2*i + 2)
        plt.imshow(trans); plt.axis("off")
        plt.title(f"{clase} — transformada (128×128)", fontsize=9)
    plt.tight_layout(); plt.show()


## 6. Conteo por clase

In [ ]:
print("Fotos por clase:")
total = 0
for clase in CLASES:
    d = os.path.join(BASE, clase)
    n = len(os.listdir(d)) if os.path.isdir(d) else 0
    total += n
    print(f"  {clase}: {n}")
print(f"  TOTAL: {total}")


## 7. Empaquetar en ZIP y descargar

In [ ]:
ruta_zip = shutil.make_archive("/content/dataset_128", "zip", "/content", "dataset")
print("ZIP creado:", ruta_zip)
files.download(ruta_zip)


## Reiniciar (opcional)

In [ ]:
shutil.rmtree(BASE, ignore_errors=True)
os.makedirs(BASE, exist_ok=True)
_previews.clear()
print("Dataset reiniciado. Podés volver a la sección 4A o 4B.")
